In [6]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [ ]:
# Import and validate Claude's API key

import os
from anthropic import Anthropic # Create client instance

claude_api_key = os.getenv('CLAUDE_API_KEY2')

if claude_api_key:
    print(f"OpenAI API Key exists and begins {claude_api_key[:8]}")
    client = Anthropic(
        api_key = claude_api_key
    )
else:
    print("OpenAI API Key not set - please head to the troubleshooting guide in the setup folder")

OpenAI API Key not set - please head to the troubleshooting guide in the setup folder


In [8]:
system_prompt = """\
<role>
You are an expert analyst covering macroeconomics, cybersecurity, technology, and financial markets, with a focus on global and Colombia. Fetch latest news and data from the web, and provide a concise, factual, and data-driven daily executive briefing. Be impartial.:
</role>

<coverage>
- Colombia macroeconomics
- USD/COP
- Cybersecurity threats, CVEs, and trends
- AI, cloud, and emerging technologies
- Stocks, ETFs, and the S&P 500
</coverage>

<rules>
- Be concise, factual, and data-driven.
- Explicitly separate facts from forecasts. Label forecasts as such.
- If you are unsure or lack current data, say so. Do not guess or fabricate figures, dates, or CVE identifiers.
- For news: cover what happened, why it matters, and the impact on Colombia.
- For markets: give both bullish and bearish factors.
- For cybersecurity: give news, global/Colombia incidents severity (CVSS if known), affected systems, and mitigations.
- Use Markdown with bullet points for scannability.
</rules>

<output_format>
Include only the sections relevant to the query (skip empty ones):
- **Executive Summary** — 128 tokens max, 2-3 sentences synthesizing the day.
- **Key Facts**
- **Impact on Colombia**
- **Risks**
- **Opportunities** - scoped to AI Engineering, CYber security, and investment opportunities.
- **Outlook / Next Events**
- **<URL>** - link to source
</output_format>

<length>
MANDATORY: Keep responses under ~2048 tokens. If a full answer would exceed this, summarize and offer to expand on specific sections rather than truncating.
</length>
"""

user_prompt =  """\
<task>
Create today's daily executive briefing using the system report structure.
</task>

<instructions>
Cover these topics, grounding every figure in current web-search data:
1. USD/COP exchange rate and main drivers.
2. Top macroeconomic news: global & Colombia.


If a data point (rate, index level, CVE) can't be verified, state that rather than estimating.
</instructions>

<output_contract>
Follow the system report structure. Map the topics into it as follows:
- **Executive Summary**: 2-3 sentences synthesizing the day.
- **Key Facts**: topics 1-6, each as a bullet —
    - Headline
    - Why it matters
    - Impact on Colombia (if applicable)
- **Impact on Colombia**: consolidated read across the topics.
- **Risks**: downside/bearish and cyber-threat factors.
- **Opportunities**: upside/bullish and strategic openings.
- **Outlook / Next Events**: topic 7, the next-24h watchlist.
- **URL**: link to source
</output_contract>
"""

In [9]:
tools = [{"type": "web_search_20250305", "name": "web_search", "max_uses": 5}]
messages=[{
    "role": "user",
    "content": user_prompt
}]

In [10]:
while True:
    with client.with_options(timeout=60.0).messages.stream(
        model="claude-opus-4-8",
        max_tokens=2048,
        system=system_prompt,
        tools=tools,
        messages=messages,
    ) as stream:
        resp = stream.get_final_message()

    if resp.stop_reason == "pause_turn":
        messages.append({"role": "assistant", "content": resp.content})
        continue
    break

answer = "".join(b.text for b in resp.content if b.type == "text")
print(answer)
print("stop_reason:", resp.stop_reason)

NameError: name 'client' is not defined